# cAPTure: Stage-2 graph-input audit

This notebook validates the boundary between the immutable FULL_DEV graph materialization and Stage-2 model code. It stages the compressed shards locally, verifies every checksum, reconstructs PyTorch Geometric graphs, and checks fold roles, chronological order, time gaps, packet-edge correspondence, and the exact global-node-ID-to-MAC mapping.

It does not train models and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Mount Drive and prepare the graph environment

Push the current branch before opening Colab. This installs PyTorch and PyTorch Geometric for graph reconstruction only; no optimization is performed.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_IDENTITY_WORK_ROOT = Path("/content/capture_graph_identity_work")

REPOSITORY_ALREADY_PRESENT = PROJECT_ROOT.exists()
if not REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
if REPOSITORY_ALREADY_PRESENT:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_graph_dataset.py",
    "code/python/utils/capture_graph_identity.py",
    "configs/capture_graph_stage2_input_v2.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import (
    audit_capture_graph_inputs,
    load_completed_capture_graph_input_audit,
    save_capture_graph_input_audit,
    stage_capture_graph_materialization,
)
from utils.capture_graph_identity import audit_capture_graph_identities

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Stage-2 graph-input environment is ready.")

Mounted at /content/drive
Repository commit: 2595e5c788282389652254f742723f37e69c6781
Stage-2 graph-input environment is ready.


## 2. Bind the immutable FULL_DEV materialization

The graph and prepared-packet run IDs are part of the input boundary. The audit refuses artifacts whose hashes differ from the graph-materialization provenance.

In [2]:
FULL_DEV_RUN_ID = "20260928T175821_521784Z_graph_materialization_full_dev"
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
SOURCE_GRAPH_ROOT = (
    DRIVE_ROOT / "graph_materialization_runs" / FULL_DEV_RUN_ID
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)

for required_path in (SOURCE_GRAPH_ROOT, PREPARED_RUN_DIR, INPUT_CONTRACT_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input is missing: {required_path}")

print("Materialization source:", SOURCE_GRAPH_ROOT)
print("Prepared packet source:", PREPARED_RUN_DIR)
print("Input contract:", INPUT_CONTRACT_PATH)

Materialization source: /content/drive/MyDrive/capture_gate0/graph_materialization_runs/20260928T175821_521784Z_graph_materialization_full_dev
Prepared packet source: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Input contract: /content/temporalgnn-nids/configs/capture_graph_stage2_input_v2.yaml


## 3. Stage and verify the graph collection

The collection is copied to `/content` so repeated graph access does not use Drive. Every shard and supporting artifact is checksum-verified after copying. A verified local copy is reused after rerunning the cell in the same runtime.

In [3]:
collection = stage_capture_graph_materialization(
    source_root=SOURCE_GRAPH_ROOT,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=INPUT_CONTRACT_PATH,
    expected_run_id=FULL_DEV_RUN_ID,
)
print("Verified local graph root:", collection.root)
print("Feature dimension:", collection.feature_dim)
print("Window duration (ms):", collection.window_ms)
print("All artifact checksums verified:",
      collection.artifact_checksums_verified)

Staged and verified graph materialization: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Verified local graph root: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Feature dimension: 103
Window duration (ms): 5000
All artifact checksums verified: True


## 4. Review the fold-role plan

A scenario has one training role and one OOF validation role across the two folds. Physical copies under both preprocessors must never be mixed inside one fold.

In [4]:
role_rows = []
for scenario in collection.scenarios:
    role_rows.append({
        "scenario": scenario,
        "fold_A": collection.partition_for("A", scenario),
        "fold_B": collection.partition_for("B", scenario),
        "OOF_fold": next(
            fold for fold in ("A", "B")
            if collection.partition_for(fold, scenario) == "validation"
        ),
    })
role_table = pd.DataFrame(role_rows).set_index("scenario")
display(role_table)

,fold_A,fold_B,OOF_fold
scenario,,,
train_empty_conn,train,validation,B
train_qos_mid,train,validation,B
train_dollar_char,validation,train,A
train_slash_char,validation,train,A
train_sub_exf,validation,train,A


## 5. Reconstruct a bounded sample

This reconstructs the first, middle, and last graph of every fold/scenario sequence. It checks tensor types and shapes, local node indexes, global node IDs, feature finiteness, binary targets, five-second timing, and the declared partition.

In [5]:
sample_report = audit_capture_graph_inputs(collection, mode="SAMPLE")
sample_rows = []
for fold, scenarios in sample_report["scenario_results"].items():
    for scenario, result in scenarios.items():
        for sample in result["sampled_graphs"]:
            sample_rows.append({
                "fold": fold,
                "scenario": scenario,
                "partition": result["partition"],
                **sample,
            })
display(pd.DataFrame(sample_rows))
print("Training edges across folds:",
      sample_report["training_edges_across_folds"])
print("OOF edges across folds:", sample_report["oof_edges_across_folds"])
print("Unique development packets:",
      sample_report["unique_development_packets"])
print("OOF scenario assignment:",
      json.dumps(sample_report["oof_scenario_fold"], indent=2))

,fold,scenario,partition,graph_position,window_index,edges,nodes,first_source_row_id,last_source_row_id,timestamp_ms
0,A,train_empty_conn,train,0,0,475,86,0,474,5019
1,A,train_empty_conn,train,4680,4680,90,15,786287,786376,23405019
2,A,train_empty_conn,train,9359,9359,81,12,1175698,1175778,46800019
3,A,train_qos_mid,train,0,0,475,86,0,474,5019
4,A,train_qos_mid,train,4680,4680,149,16,782073,782221,23405019
5,A,train_qos_mid,train,9359,9359,81,12,1499636,1499716,46800019
6,A,train_dollar_char,validation,0,0,476,85,0,475,5009
7,A,train_dollar_char,validation,4558,4559,603,16,1735580,1736182,22800009
8,A,train_dollar_char,validation,9115,9359,2,2,2882553,2882554,46800009
9,A,train_slash_char,validation,0,0,476,85,0,475,5009


Training edges across folds: 14209374
OOF edges across folds: 14209374
Unique development packets: 14209374
OOF scenario assignment: {
  "train_dollar_char": "A",
  "train_slash_char": "A",
  "train_sub_exf": "A",
  "train_empty_conn": "B",
  "train_qos_mid": "B"
}


## 6. Manual approval for the full loader scan

Approve only if the role table and 30 sampled graphs are plausible, the three packet totals are equal, and all artifact checksums were verified. The full scan reconstructs every graph representation but still performs no model training.

In [6]:
APPROVE_FULL_INPUT_AUDIT = True
SAMPLE_REVIEW_NOTE = (
    "Fold roles, sampled graph boundaries, checksum verification, and packet "
    "coverage were reviewed. Training and OOF edges both match the "
    "14,209,374 unique development packets."
)

if not APPROVE_FULL_INPUT_AUDIT:
    raise RuntimeError(
        "Review the sample, then set APPROVE_FULL_INPUT_AUDIT=True."
    )
if not SAMPLE_REVIEW_NOTE.strip():
    raise ValueError("Record a non-empty SAMPLE_REVIEW_NOTE.")
if not collection.artifact_checksums_verified:
    raise RuntimeError("Every staged artifact checksum must be verified.")
print(SAMPLE_REVIEW_NOTE)

Fold roles, sampled graph boundaries, checksum verification, and packet coverage were reviewed. Training and OOF edges both match the 14,209,374 unique development packets.


## 7. Run or resume the full graph-input audit

The full audit verifies contiguous `source_row_id` values and strictly increasing windows and timestamps for every graph sequence. It then joins every graph edge to the exact prepared packet, reconstructs a bijective global-node-ID-to-MAC mapping, and checks its semantic hash against both folds. The diagnostic MAC lookups are stored under the audit run and are never model inputs. Preserve the printed audit run ID.

In [7]:
FULL_INPUT_AUDIT_RUN_ID = None  # Paste an existing completed ID to reuse it.
if FULL_INPUT_AUDIT_RUN_ID is None:
    FULL_INPUT_AUDIT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_input_full"
    )
FULL_INPUT_AUDIT_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / FULL_INPUT_AUDIT_RUN_ID
)
print("FULL input-audit run ID:", FULL_INPUT_AUDIT_RUN_ID)
print("FULL input-audit output:", FULL_INPUT_AUDIT_OUTPUT_DIR)

if FULL_INPUT_AUDIT_OUTPUT_DIR.exists():
    full_report = load_completed_capture_graph_input_audit(
        FULL_INPUT_AUDIT_OUTPUT_DIR, collection, mode="FULL"
    )
    print("Reused the completed immutable input audit.")
else:
    full_report = audit_capture_graph_inputs(collection, mode="FULL")
    identity_result = audit_capture_graph_identities(
        collection,
        prepared_run_dir=PREPARED_RUN_DIR,
        local_work_root=LOCAL_IDENTITY_WORK_ROOT,
    )
    full_report["identity_audit"] = identity_result["report"]
    full_report["sample_review_note"] = SAMPLE_REVIEW_NOTE
    save_capture_graph_input_audit(
        FULL_INPUT_AUDIT_OUTPUT_DIR,
        full_report,
        contract_path=INPUT_CONTRACT_PATH,
        identity_lookups=identity_result["lookups"],
    )
    print("Saved the completed immutable input audit.")

FULL input-audit run ID: 20260928T193308_017682Z_graph_input_full
FULL input-audit output: /content/drive/MyDrive/capture_gate0/graph_input_audit_runs/20260928T193308_017682Z_graph_input_full
Recovered train_empty_conn: 160 node identities from 1,175,779 edges
Recovered train_qos_mid: 163 node identities from 1,499,717 edges
Recovered train_dollar_char: 160 node identities from 2,882,555 edges
Recovered train_slash_char: 164 node identities from 6,425,516 edges
Recovered train_sub_exf: 121 node identities from 2,225,807 edges
Saved the completed immutable input audit.


## 8. Review the full input boundary

Every row must report a complete sequence with contiguous source rows and increasing windows/timestamps. Every identity row must report a conflict-free bijection whose reconstructed semantic hash matches both folds. Passing this gate authorizes implementation of the frozen one-seed training runner; it is not model evidence.

In [8]:
full_rows = []
for fold, scenarios in full_report["scenario_results"].items():
    for scenario, result in scenarios.items():
        full_rows.append(result)
full_table = pd.DataFrame(full_rows)
display(full_table)
identity_rows = []
for scenario, result in full_report["identity_audit"]["scenarios"].items():
    identity_rows.append({"scenario": scenario, **result})
identity_table = pd.DataFrame(identity_rows)
display(identity_table)
print("Training edges across folds:",
      full_report["training_edges_across_folds"])
print("OOF edges across folds:", full_report["oof_edges_across_folds"])
print("Unique development packets:",
      full_report["unique_development_packets"])
print("Held-out scenarios accessed:",
      full_report["held_out_scenarios_accessed"])
print("Training performed:", full_report["training_performed"])
print("Audit report:",
      FULL_INPUT_AUDIT_OUTPUT_DIR / "capture_graph_input_audit.json")
print("Diagnostic identity lookups:",
      FULL_INPUT_AUDIT_OUTPUT_DIR / "node_identity_lookup")

required_flags = [
    "complete_sequence",
    "source_rows_contiguous",
    "windows_and_timestamps_strictly_increasing",
]
if not full_table[required_flags].all(axis=None):
    raise RuntimeError("The full graph-input boundary audit failed.")
identity_required = [
    "source_rows_contiguous",
    "first_seen_rows_match_stored_mapping",
    "reconstructed_contract_matches_both_folds",
]
if not identity_table[identity_required].all(axis=None):
    raise RuntimeError("The node-identity boundary audit failed.")
if not (
    identity_table["id_to_mac_conflicts"].eq(0).all()
    and identity_table["mac_to_id_conflicts"].eq(0).all()
    and identity_table["unresolved_global_nodes"].eq(0).all()
):
    raise RuntimeError("The recovered node mapping is not bijective.")
if not (
    full_report["training_edges_across_folds"]
    == full_report["oof_edges_across_folds"]
    == full_report["unique_development_packets"]
):
    raise RuntimeError("Fold roles do not cover each packet exactly once.")

,fold,scenario,partition,complete_sequence,graphs,edges,attack_edges,node_appearances,maximum_edges_in_graph,first_source_row_id,last_source_row_id,first_window_index,last_window_index,source_rows_contiguous,windows_and_timestamps_strictly_increasing,wall_seconds,graphs_per_second,edges_per_second
0,A,train_empty_conn,train,True,9360,1175779,428973,126324,42145,0,1175778,0,9359,True,True,3.463635,2702.363587,339463.926915
1,A,train_qos_mid,train,True,9360,1499717,752911,127220,42191,0,1499716,0,9359,True,True,3.625621,2581.626735,413644.177520
2,A,train_dollar_char,validation,True,9116,2882555,1177552,116773,43177,0,2882554,0,9359,True,True,5.593017,1629.889694,515384.674007
3,A,train_slash_char,validation,True,9116,6425516,4720513,115510,42517,0,6425515,0,9359,True,True,8.287598,1099.956810,775317.033863
4,A,train_sub_exf,validation,True,9115,2225807,520804,117843,42250,0,2225806,0,9359,True,True,4.866418,1873.041082,457381.014946
5,B,train_empty_conn,validation,True,9360,1175779,428973,126324,42145,0,1175778,0,9359,True,True,3.270765,2861.715935,359481.356888
6,B,train_qos_mid,validation,True,9360,1499717,752911,127220,42191,0,1499716,0,9359,True,True,3.538313,2645.328537,423850.873671
7,B,train_dollar_char,train,True,9116,2882555,1177552,116773,43177,0,2882554,0,9359,True,True,5.506757,1655.420673,523457.781817
8,B,train_slash_char,train,True,9116,6425516,4720513,115510,42517,0,6425515,0,9359,True,True,8.430072,1081.366791,762213.648017
9,B,train_sub_exf,train,True,9115,2225807,520804,117843,42250,0,2225806,0,9359,True,True,4.852253,1878.508682,458716.157405


,scenario,status,prepared_packet_sha256,canonical_graph_fold,edges_joined,global_nodes_resolved,unique_mac_addresses,id_to_mac_conflicts,mac_to_id_conflicts,unresolved_global_nodes,source_rows_contiguous,first_seen_rows_match_stored_mapping,reconstructed_mapping_contract_sha256,fold_mapping_contract_sha256,reconstructed_contract_matches_both_folds,wall_seconds,lookup_artifact,lookup_sha256
0,train_empty_conn,passed,49ee5c10a915117c802583ee915e474f93b3bffa85a820...,A,1175779,160,160,0,0,0,True,True,88a80fc212c040c67ddf36631b074e4e5ebd70bbcf73e7...,{'A': '88a80fc212c040c67ddf36631b074e4e5ebd70b...,True,0.871372,node_identity_lookup/train_empty_conn.parquet,1e3475473372efa5444803e9543747e907b45906e3784c...
1,train_qos_mid,passed,b43cde914424f7a2fe7f5df7c0a205dfa6052e45817783...,A,1499717,163,163,0,0,0,True,True,b99a7d990e8e8b5b8af46250197e2da1421b0e69b2a878...,{'A': 'b99a7d990e8e8b5b8af46250197e2da1421b0e6...,True,0.865043,node_identity_lookup/train_qos_mid.parquet,8f4e97ff69a18499e5891c8b878f710f354cf5a1ce0de8...
2,train_dollar_char,passed,2b7100c5ba61497de39d83b6b96a223096f936a3a09dba...,A,2882555,160,160,0,0,0,True,True,82c7c54d5d13ab3c1982e4cab2f0bf0376353970ee58c2...,{'A': '82c7c54d5d13ab3c1982e4cab2f0bf037635397...,True,1.390665,node_identity_lookup/train_dollar_char.parquet,3ea2f0c8e73bde0fd9a5d7a71f8f2a46e0dc0cb3f3d7ac...
3,train_slash_char,passed,aa6799cb01c688fd6e44a32e3473fda7b5f7cb7a232b3b...,A,6425516,164,164,0,0,0,True,True,368b312a8bc32d54b07fd2a2867df7d6010c94db1080d3...,{'A': '368b312a8bc32d54b07fd2a2867df7d6010c94d...,True,3.396584,node_identity_lookup/train_slash_char.parquet,ec960119af61cfea378b648c2a084ba857b405262f2abf...
4,train_sub_exf,passed,b4ec7570a55964ab4f1ccad2d41a7e780a26deae7f6af4...,A,2225807,121,121,0,0,0,True,True,4280788fe875c0ee89bf021d4390d908cbc9fae57d9b4d...,{'A': '4280788fe875c0ee89bf021d4390d908cbc9fae...,True,1.601120,node_identity_lookup/train_sub_exf.parquet,d4fb237d46c9246c3726d52dc142f49ca25fc6d45d8685...


Training edges across folds: 14209374
OOF edges across folds: 14209374
Unique development packets: 14209374
Held-out scenarios accessed: False
Training performed: False
Audit report: /content/drive/MyDrive/capture_gate0/graph_input_audit_runs/20260928T193308_017682Z_graph_input_full/capture_graph_input_audit.json
Diagnostic identity lookups: /content/drive/MyDrive/capture_gate0/graph_input_audit_runs/20260928T193308_017682Z_graph_input_full/node_identity_lookup


In [9]:
identity_directory = (
    FULL_INPUT_AUDIT_OUTPUT_DIR / "node_identity_lookup"
)

empty_conn_lookup = pd.read_parquet(
    identity_directory / "train_empty_conn.parquet"
)

display(empty_conn_lookup)

,scenario,global_node_id,mac_address,first_seen_source_row_id,first_seen_role
0,train_empty_conn,0,96:a5:fa:53:10:d0,0,source
1,train_empty_conn,1,33:33:00:00:00:16,0,destination
2,train_empty_conn,2,fa:be:65:a8:61:93,1,source
3,train_empty_conn,3,33:33:ff:a8:61:93,2,destination
4,train_empty_conn,4,fe:73:58:53:d6:db,3,source
...,...,...,...,...,...
155,train_empty_conn,155,6a:db:6d:35:cf:81,21549,source
156,train_empty_conn,156,36:14:bb:67:13:83,21568,source
157,train_empty_conn,157,ce:14:54:91:ef:63,21575,source
158,train_empty_conn,158,12:34:84:a7:56:ea,21579,source
